# 🎬 MovieIQ - Predictive Analytics on Film Success

## Objective
The objective of this project is to analyze movie data, identify the factors influencing movie success, perform statistical analysis, build a machine learning model to predict movie success, and deploy the solution using Streamlit.

---

### Workflow

1. Import Libraries
2. Load Dataset
3. Data Understanding
4. Data Cleaning
5. Exploratory Data Analysis (EDA)
6. Statistical Testing
7. Machine Learning Model
8. Streamlit Dashboard

In [133]:
# Import Libraries

# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Statistical Analysis
from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

# Save Model
import joblib

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

In [134]:
# Load Dataset

df = pd.read_csv("movies.csv")

In [135]:
# Display first five rows

df.head()

,budget,revenue,popularity,runtime,vote_average,title,genres,success,roi
0,57755036,107323371,60.377360,98,6.611888,Movie 1,Romance,1,1.858251
1,192100010,342896533,86.289857,96,7.184687,Movie 2,Drama,1,1.784990
2,128521863,253206827,86.995324,148,3.542978,Movie 3,Comedy,1,1.970146
3,156299516,89635602,72.710096,118,5.985176,Movie 4,Drama,0,0.573486
4,148532820,221133868,74.829585,161,4.578480,Movie 5,Comedy,0,1.488788


In [136]:
# Dataset Shape

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 1819
Columns : 9


In [137]:
# Dataset Information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1819 entries, 0 to 1818
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   budget        1819 non-null   int64  
 1   revenue       1819 non-null   int64  
 2   popularity    1819 non-null   float64
 3   runtime       1819 non-null   int64  
 4   vote_average  1819 non-null   float64
 5   title         1819 non-null   str    
 6   genres        1819 non-null   str    
 7   success       1819 non-null   int64  
 8   roi           1819 non-null   float64
dtypes: float64(3), int64(4), str(2)
memory usage: 158.9 KB


In [138]:
# Summary Statistics

df.describe()

,budget,revenue,popularity,runtime,vote_average,success,roi
count,1.819000e+03,1.819000e+03,1819.000000,1819.000000,1819.000000,1819.000000,1819.000000
mean,1.023851e+08,1.782876e+08,50.273368,129.476636,6.023922,0.606377,1.751267
std,5.670612e+07,1.287606e+08,28.387060,28.906851,1.740184,0.488687,0.713292
min,1.797968e+06,1.994306e+06,1.018466,80.000000,3.009477,0.000000,0.500471
25%,5.339322e+07,7.493540e+07,26.006233,104.000000,4.567721,0.000000,1.135570
50%,1.029189e+08,1.492668e+08,50.228135,129.000000,6.051165,1.000000,1.767096
75%,1.510351e+08,2.621539e+08,75.033830,155.000000,7.548113,1.000000,2.368835
max,1.999879e+08,5.744901e+08,99.764659,179.000000,8.998509,1.000000,2.999285


In [139]:
# Column Names

df.columns

Index(['budget', 'revenue', 'popularity', 'runtime', 'vote_average', 'title',
       'genres', 'success', 'roi'],
      dtype='str')

In [140]:
# Data Types

df.dtypes

budget            int64
revenue           int64
popularity      float64
runtime           int64
vote_average    float64
title               str
genres              str
success           int64
roi             float64
dtype: object

In [141]:
# Missing Values

df.isnull().sum()

# Missing Value Percentage

missing = (df.isnull().sum()/len(df))*100

missing.sort_values(ascending=False)

budget          0.0
revenue         0.0
popularity      0.0
runtime         0.0
vote_average    0.0
title           0.0
genres          0.0
success         0.0
roi             0.0
dtype: float64

In [142]:
# Duplicate Rows

df.duplicated().sum()

np.int64(0)

In [143]:
# Budget should be greater than 0
(df["budget"] <= 0).sum()

# Revenue should be greater than 0
(df["revenue"] <= 0).sum()

# Runtime should be greater than 0
(df["runtime"] <= 0).sum()

# Vote Average should be between 0 and 10
((df["vote_average"] < 0) | (df["vote_average"] > 10)).sum()

np.int64(0)

In [144]:
df["genres"].head(10)

0      Romance
1        Drama
2       Comedy
3        Drama
4       Comedy
5      Romance
6    Adventure
7       Action
8       Horror
9       Horror
Name: genres, dtype: str

In [145]:
(df["genres"] == "[]").sum()

np.int64(0)

## Cleaning the `genres` Column

The dataset contains no null values. However, 181 records contain empty lists (`[]`) in the `genres` column, indicating that genre information is unavailable.

Since genre is required for exploratory data analysis, statistical testing, and dashboard filtering, these records are removed before further analysis.

In [146]:
# Remove movies with empty genres

df = df[df["genres"] != "[]"]

# Reset index

df.reset_index(drop=True, inplace=True)

# Check new shape

df.shape

(1819, 9)

In [147]:
df.to_csv("movies.csv", index=False)

In [148]:
check = pd.read_csv("movies.csv")
print(check["genres"].head())

0    Romance
1      Drama
2     Comedy
3      Drama
4     Comedy
Name: genres, dtype: str


In [149]:
df["genres"].unique()

<ArrowStringArray>
[        'Romance',           'Drama',          'Comedy',       'Adventure',
          'Action',          'Horror',       'Animation',        'Thriller',
 'Science Fiction']
Length: 9, dtype: str

# Feature Engineering

Feature engineering is the process of creating new features from the existing dataset that can improve analysis and machine learning performance.

For this project, a new target variable named **success** is created based on the following rule:

- **Success (1):** Revenue > Budget
- **Failure (0):** Revenue ≤ Budget

This target variable will be used for machine learning classification.

In [203]:
# Create Target Variable (Success)

df["success"] = (df["revenue"] > df["revenue"].median()).astype(int)

In [151]:
# Count Success and Failure

df["success"].value_counts()

success
0    910
1    909
Name: count, dtype: int64

In [152]:
X = df[[
    "budget",
    "popularity",
    "runtime",
    "vote_average",
    "genres"
]]

y = df["success"]

In [153]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [154]:
# Percentage of Success and Failure

(df["success"].value_counts(normalize=True) * 100).round(2)

success
0    50.03
1    49.97
Name: proportion, dtype: float64

In [155]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1819 entries, 0 to 1818
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   budget        1819 non-null   int64  
 1   revenue       1819 non-null   int64  
 2   popularity    1819 non-null   float64
 3   runtime       1819 non-null   int64  
 4   vote_average  1819 non-null   float64
 5   title         1819 non-null   str    
 6   genres        1819 non-null   str    
 7   success       1819 non-null   int64  
 8   roi           1819 non-null   float64
dtypes: float64(3), int64(4), str(2)
memory usage: 158.9 KB


## Feature Engineering Summary

A new binary target variable named **success** was created using the movie's budget and revenue.

- **1** indicates the movie earned more revenue than its production budget.
- **0** indicates the movie failed to recover its production budget.

This target variable will be used to train the machine learning classification model.

In [156]:
# Save cleaned dataset

df.to_csv("movies_cleaned.csv", index=False)

# Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps understand the characteristics of the dataset using statistical summaries and visualizations.

The objective is to identify trends, relationships, and patterns that influence movie success.

## 1. Movie Success Distribution

This chart shows the number of successful and unsuccessful movies in the dataset.

In [201]:
import plotly.express as px

success_count = df["success"].value_counts().reset_index()
success_count.columns = ["Success", "Count"]

success_count["Success"] = success_count["Success"].replace({
    1: "Success",
    0: "Failure"
})

fig = px.bar(
    success_count,
    x="Success",
    y="Count",
    color="Success",
    text="Count",
    title="Movie Success Distribution"
)

fig.update_layout(
    template="plotly_dark",
    title_x=0.5,
    xaxis_title="Movie Status",
    yaxis_title="Number of Movies",
    showlegend=False
)

fig.show()

### Insight

- Successful movies are significantly higher than unsuccessful movies.
- Around 81% of the movies recovered their production budget.
- This indicates that the dataset is imbalanced, which should be considered during machine learning model evaluation.

## 2. Budget vs Revenue

This scatter plot visualizes the relationship between a movie's production budget and its revenue. It helps identify whether higher-budget movies tend to generate higher revenue.

In [158]:
fig = px.scatter(
    df,
    x="budget",
    y="revenue",
    color="success",
    hover_data=["title", "genres"],
    title="Budget vs Revenue",
    color_discrete_map={
        1: "#00BFFF",   # Electric Blue
        0: "#FF6B6B"    # Soft Red
    }
)

fig.update_layout(
    template="plotly_dark",
    title_x=0.5,
    xaxis_title="Budget",
    yaxis_title="Revenue",
    legend_title="Success"
)

fig.show()

### Insight

- Movies with larger budgets generally tend to earn higher revenue.
- However, a high budget does not always guarantee success.
- Some low-budget movies also achieve high revenue, indicating that factors such as popularity and audience ratings also influence success.

## 3. Genre Distribution

This chart shows the number of movies in each genre. It helps identify the most common genres in the dataset and supports genre-based filtering in the dashboard.

In [159]:
genre_count = (
    df["genres"]
    .value_counts()
    .reset_index()
)

genre_count.columns = ["Genre", "Count"]

In [160]:
fig = px.bar(
    genre_count,
    x="Genre",
    y="Count",
    text="Count",
    color="Count",
    color_continuous_scale=[
        "#1565C0",  # Dark Blue
        "#4FC3F7",  # Electric Blue
        "#8ED1FC",  # Sky Blue
        "#D6F4FF"   # Light Cyan
    ],
    title="Genre Distribution"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white", size=13),
    title=dict(text="Genre Distribution", x=0.5),
    xaxis_title="Genre",
    yaxis_title="Number of Movies",
    coloraxis_showscale=False,
    height=550
)

fig.update_traces(
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Movies: %{y}<extra></extra>"
)

fig.show()

### Insight

- The chart shows the distribution of movies across different genres.
- Genres with taller bars are more frequently represented in the dataset.
- This information helps identify dominant genres and supports genre-based filtering in the Streamlit dashboard.

## 4. Average Revenue by Genre

This chart shows the average revenue generated by each movie genre. It helps identify which genres are the most profitable.

In [161]:
# Calculate average revenue for each genre
genre_revenue = (
    df.groupby("genres")["revenue"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

import plotly.express as px

fig = px.bar(
    genre_revenue,
    x="genres",
    y="revenue",
    text_auto=".2s",
    color="revenue",
    color_continuous_scale=[
        "#1565C0",  # Dark Blue
        "#4FC3F7",  # Electric Blue
        "#8ED1FC",  # Sky Blue
        "#D6F4FF"   # Light Cyan
    ],
    title="Average Revenue by Genre"
)

fig.update_traces(
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Average Revenue: %{y:,.0f}<extra></extra>"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white", size=13),
    title=dict(
        text="Average Revenue by Genre",
        x=0.5
    ),
    xaxis_title="Genre",
    yaxis_title="Average Revenue",
    coloraxis_showscale=False,
    height=550
)

fig.show()

### Insight

- This chart compares the average revenue earned by each movie genre.
- Genres with taller bars generate higher average revenue than others.
- These insights help identify which genres are financially more successful and may influence future production decisions.

## 5. Popularity vs Revenue

This scatter plot shows the relationship between a movie's popularity and its revenue. It helps determine whether popular movies tend to earn higher revenue.

In [162]:
fig = px.scatter(
    df,
    x="popularity",
    y="revenue",
    color="success",
    hover_name="title",
    hover_data=["genres", "budget", "vote_average"],
    color_discrete_map={
         1: "#4FC3F7",
         0: "#EF5350"
  # Soft Red
    },
    title="Popularity vs Revenue"
)

fig.update_traces(
    marker=dict(
        size=9,
        opacity=0.75,
        line=dict(width=1, color="white")
    )
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white", size=13),
    title=dict(
        text="Popularity vs Revenue",
        x=0.5
    ),
    xaxis_title="Popularity Score",
    yaxis_title="Revenue",
    legend_title="Movie Status",
    height=550
)

fig.show()

### Insight

- Movies with higher popularity generally tend to earn higher revenue.
- However, popularity alone does not guarantee financial success.
- This indicates that popularity is an important feature but should be considered alongside budget, runtime, vote average, and genre when predicting movie success.

## 6. Runtime Distribution

This histogram shows how movie runtimes are distributed across the dataset. It helps identify the most common movie duration and detect unusually short or long movies.

In [163]:
import plotly.express as px

fig = px.histogram(
    df,
    x="runtime",
    nbins=25,
    title="Distribution of Movie Runtime"
)

fig.update_traces(
    marker=dict(
        color="#4FC3F7",      # Electric Blue
        line=dict(color="white", width=1)
    ),
    hovertemplate="Runtime: %{x} mins<br>Movies: %{y}<extra></extra>"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white", size=13),
    title=dict(
        text="Distribution of Movie Runtime",
        x=0.5
    ),
    xaxis_title="Runtime (Minutes)",
    yaxis_title="Number of Movies",
    height=550
)

fig.show()

### Insight

- The histogram shows the distribution of movie runtimes.
- Most movies are concentrated within a specific runtime range.
- Extremely short or very long movies are relatively uncommon.
- Runtime can be an important feature when analysing movie characteristics and predicting success.

## 7. Vote Average Distribution

This histogram illustrates the distribution of movie ratings (`vote_average`) in the dataset. It helps understand the overall rating pattern and identify common rating ranges.

In [164]:
import plotly.express as px

fig = px.histogram(
    df,
    x="vote_average",
    nbins=20,
    title="Distribution of Movie Ratings"
)

fig.update_traces(
    marker=dict(
        color="#4FC3F7",          # Electric Blue
        line=dict(color="white", width=1)
    ),
    hovertemplate="Rating: %{x:.1f}<br>Movies: %{y}<extra></extra>"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white", size=13),
    title=dict(
        text="Distribution of Movie Ratings",
        x=0.5
    ),
    xaxis_title="Vote Average",
    yaxis_title="Number of Movies",
    height=550
)

fig.show()

### Insight

- The histogram shows how movie ratings are distributed across the dataset.
- Most movies are concentrated within a particular rating range.
- Very low-rated and very high-rated movies are relatively uncommon.
- Movie ratings may influence audience interest and can contribute to predicting movie success.

## 8. Correlation Heatmap

The correlation heatmap visualizes the relationship between numerical features in the dataset. It helps identify positive and negative correlations, which are useful for feature selection in machine learning.

In [165]:
import plotly.express as px

# Select only numerical columns
corr = df[[
    "budget",
    "revenue",
    "popularity",
    "runtime",
    "vote_average",
    "success"
]].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale=[
        "#0D47A1",   # Dark Blue
        "#42A5F5",   # Blue
        "#E3F2FD"    # Light Blue
    ],
    title="Correlation Heatmap"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white", size=13),
    title=dict(
        text="Correlation Heatmap",
        x=0.5
    ),
    height=600
)

fig.show()

### Insight

- The heatmap displays the correlation between all numerical features.
- Correlation values closer to **1** indicate a strong positive relationship.
- Correlation values closer to **-1** indicate a strong negative relationship.
- Values near **0** indicate little or no linear relationship.
- The `success` column helps identify which features are most closely associated with movie success and are therefore useful for machine learning.

In [166]:
from scipy.stats import ttest_ind, chi2_contingency
import pandas as pd

# Statistical Analysis

Statistical tests are performed to validate whether the observed relationships in the dataset are statistically significant before building the machine learning model.

The following tests are used:

- Independent T-Test
- Chi-Square Test

## 1. Independent T-Test

The Independent T-Test compares the average movie ratings (`vote_average`) between successful and unsuccessful movies.

### Hypotheses

- **Null Hypothesis (H₀):** There is no significant difference in the average ratings of successful and unsuccessful movies.
- **Alternative Hypothesis (H₁):** There is a significant difference in the average ratings.

In [167]:
# Separate ratings based on success

success_rating = df[df["success"] == 1]["vote_average"]
failure_rating = df[df["success"] == 0]["vote_average"]

# Perform T-Test

t_stat, p_value = ttest_ind(success_rating, failure_rating)

print("T-Statistic :", round(t_stat, 4))
print("P-Value :", round(p_value, 4))

T-Statistic : -0.6084
P-Value : 0.543


In [168]:
alpha = 0.05

if p_value < alpha:
    print("Reject the Null Hypothesis")
else:
    print("Fail to Reject the Null Hypothesis")

Fail to Reject the Null Hypothesis


### Interpretation

If the p-value is less than 0.05, the difference in average movie ratings between successful and unsuccessful movies is statistically significant.

Otherwise, there is insufficient evidence to conclude that movie ratings differ significantly based on success.

## 2. Chi-Square Test

The Chi-Square Test determines whether movie genre is associated with movie success.

### Hypotheses

- **Null Hypothesis (H₀):** Genre and movie success are independent.
- **Alternative Hypothesis (H₁):** Genre and movie success are associated.

In [169]:
# Create contingency table

contingency_table = pd.crosstab(df["genres"], df["success"])

contingency_table

success,0,1
genres,,
Action,97,95
Adventure,99,108
Animation,97,106
Comedy,102,100
Drama,101,96
Horror,92,107
Romance,110,110
Science Fiction,110,95
Thriller,102,92


In [170]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-Square Statistic :", round(chi2, 4))
print("P-Value :", round(p_value, 4))
print("Degrees of Freedom :", dof)

Chi-Square Statistic : 3.701
P-Value : 0.883
Degrees of Freedom : 8


In [171]:
alpha = 0.05

if p_value < alpha:
    print("Reject the Null Hypothesis")
else:
    print("Fail to Reject the Null Hypothesis")

Fail to Reject the Null Hypothesis


### Interpretation

If the p-value is less than 0.05, movie genre and movie success are significantly associated.

If the p-value is greater than 0.05, there is no significant association between genre and movie success.

# Machine Learning Model

The objective of this model is to predict whether a movie will be successful or unsuccessful based on its characteristics.

Since the target variable contains two classes (Success and Failure), classification algorithms are used.

In [172]:
# Create ML dataset

ml_df = df[
    [
        "budget",
        "popularity",
        "runtime",
        "vote_average",
        "genres",
        "success"
    ]
]

ml_df.head()

,budget,popularity,runtime,vote_average,genres,success
0,57755036,60.377360,98,6.611888,Romance,0
1,192100010,86.289857,96,7.184687,Drama,1
2,128521863,86.995324,148,3.542978,Comedy,1
3,156299516,72.710096,118,5.985176,Drama,0
4,148532820,74.829585,161,4.578480,Comedy,1


In [173]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore")

genre_encoded = encoder.fit_transform(
    ml_df[["genres"]]
)

In [174]:
X = ml_df.drop("success", axis=1)
y = ml_df["success"]

In [175]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Data Preprocessing

Before training the machine learning model, the dataset is preprocessed.

The preprocessing steps include:
- Separating features and target
- Encoding categorical variables
- Splitting the dataset into training and testing sets
- Creating a preprocessing pipeline

In [176]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [177]:
# Features
X = df[[
    "budget",
    "popularity",
    "runtime",
    "vote_average",
    "genres"
]]

# Target
y = df["success"]

print("Features Shape :", X.shape)
print("Target Shape :", y.shape)

Features Shape : (1819, 5)
Target Shape : (1819,)


In [178]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data :", X_train.shape)
print("Testing Data :", X_test.shape)

Training Data : (1455, 5)
Testing Data : (364, 5)


In [179]:
# Numerical Features
numeric_features = [
    "budget",
    "popularity",
    "runtime",
    "vote_average"
]

# Categorical Feature
categorical_features = [
    "genres"
]

In [180]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [181]:
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_estimators=200,
    max_depth=10
))
])

In [205]:
pipeline.fit(X_train, y_train)

import joblib
joblib.dump(pipeline, "model.pkl")

['model.pkl']

In [183]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['budget','popularity','runtime','vote_average','genres']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwer

In [184]:
y_pred = pipeline.predict(X_test)

# Model Evaluation

The trained Random Forest model is evaluated using different performance metrics to measure its prediction accuracy and reliability.

The following evaluation metrics are used:

- Accuracy
- Precision
- Recall
- F1-Score
- Confusion Matrix
- Classification Report

In [185]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [186]:
# Predict on test data

y_pred = pipeline.predict(X_test)

In [206]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.2%}")

Accuracy : 80.49%


In [188]:
precision = precision_score(y_test, y_pred)

print(f"Precision : {precision:.2%}")

Precision : 73.62%


In [189]:
recall = recall_score(y_test, y_pred)

print(f"Recall : {recall:.2%}")

Recall : 95.05%


In [190]:
f1 = f1_score(y_test, y_pred)

print(f"F1 Score : {f1:.2%}")

F1 Score : 82.97%


In [191]:
print("Accuracy :", round(accuracy, 4))
print("Precision :", round(precision, 4))
print("Recall :", round(recall, 4))
print("F1 Score :", round(f1, 4))

Accuracy : 0.8049
Precision : 0.7362
Recall : 0.9505
F1 Score : 0.8297


In [192]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.66      0.77       182
           1       0.74      0.95      0.83       182

    accuracy                           0.80       364
   macro avg       0.83      0.80      0.80       364
weighted avg       0.83      0.80      0.80       364



In [193]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[120  62]
 [  9 173]]


In [194]:
import plotly.express as px

fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale=[
        "#0D47A1",
        "#42A5F5",
        "#E3F2FD"
    ],
    labels=dict(x="Predicted", y="Actual"),
    x=["Failure", "Success"],
    y=["Failure", "Success"],
    title="Confusion Matrix"
)

fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1E1E1E",
    plot_bgcolor="#2D2D30",
    font=dict(color="white"),
    title=dict(x=0.5),
    height=500
)

fig.show()

## Model Evaluation Summary

The Random Forest classifier was evaluated using multiple performance metrics.

- **Accuracy** measures the overall prediction performance.
- **Precision** indicates how many predicted successful movies were actually successful.
- **Recall** measures how many actual successful movies were correctly identified.
- **F1-Score** provides the balance between precision and recall.
- **Confusion Matrix** visualizes correct and incorrect predictions.

These metrics help determine whether the model is suitable for deployment in the Streamlit application.

## Saving the Trained Model

The trained machine learning pipeline is saved using Joblib. The saved model includes both the preprocessing steps and the Random Forest classifier, allowing it to be loaded directly in the Streamlit application for predictions.

In [210]:
import joblib
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, "model.pkl")
model = joblib.load("model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [196]:
df.to_csv("movies.csv", index=False)

In [197]:
pred = pipeline.predict(X_test)

import pandas as pd
print(pd.Series(pred).value_counts())

1    235
0    129
Name: count, dtype: int64


In [198]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

[[120  62]
 [  9 173]]
